In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('cleaned_data.csv').drop("Unnamed: 0", axis=1)

In [4]:
pd.set_option('display.max_rows', None)

pd.set_option('display.max_columns', None)

pd.set_option('display.width', None)

pd.set_option('display.max_colwidth', None)

In [5]:
df.shape

(103, 22)

In [6]:
df_encoded = pd.get_dummies(df, columns=['3D-model', 'rarity', 'weapon', 'element', 'region', 'asc_stat_bonus_category'], prefix=['model', 'rarity', 'weapon', 'element', 'region', 'asc_bonus'], dtype=int)

In [7]:
df_encoded.shape

(103, 60)

In [8]:
df_encoded.drop(['rarity_4', 'weapon_Sword', 'element_Pyro', 'region_Extra-Terresterial', 'asc_bonus_Pyro DMG Bonus', 'model_Tall Male'], axis = 1, inplace=True)
df_encoded = df_encoded.rename(columns={'pulled_count_TARGET_VARIABLE': 'pulled_count', 'rarity_5': 'rarity_5_TARGET_VARIABLE'})
cols = ['rarity_5_TARGET_VARIABLE'] + [col for col in df_encoded.columns if col != 'rarity_5_TARGET_VARIABLE']
df_encoded = df_encoded[cols]
df_encoded['ones'] = 1

In [9]:
df_encoded['pulled_count'] = np.log10(df_encoded['pulled_count']+1)

columns_to_normalize = ['pulled_count', 'lvl_90_HP', 'lvl_90_ATK', 'lvl_90_DEF', 'months_since_release', 'num_banners', 'avg_copies_per_player']
for col in columns_to_normalize:
    df_encoded[col] = (df_encoded[col] - df_encoded[col].min()) / (df_encoded[col].max() - df_encoded[col].min())

In [10]:
df_target = df_encoded['rarity_5_TARGET_VARIABLE']
df_inputs = df_encoded.drop('rarity_5_TARGET_VARIABLE', axis=1)

In [11]:
print(df_target.shape)
print(df_inputs.shape)

(103,)
(103, 54)


In [15]:
values = np.random.randn(2, 54) * 0.01
biases = np.zeros(2)

loss_last = float('inf')
result = pd.DataFrame()
result['Linear1'] = (df_inputs * values[0]).sum(axis=1) + biases[0]
result['Linear2'] = (df_inputs * values[1]).sum(axis=1) + biases[1]
result['ReLU1'] = result['Linear1'].apply(lambda x: max(0, x))
result['ReLU2'] = result['Linear2'].apply(lambda x: max(0, x))
result['Preds'] = result['ReLU1'] + result['ReLU2']
result['Actual'] = df_target
result['Loss'] = (result['Preds'] - result['Actual'])**2
loss_last = result['Loss'].mean()

print(f'Initial loss: {loss_last}')

Initial loss: 0.5491193122409506


In [16]:
result

,Linear1,Linear2,ReLU1,ReLU2,Preds,Actual,Loss
0,-0.022530,-0.002744,0.000000,0.000000,0.000000,0,0.000000e+00
1,-0.003113,0.024690,0.000000,0.024690,0.024690,1,9.512290e-01
2,-0.017007,-0.026699,0.000000,0.000000,0.000000,0,0.000000e+00
3,0.027894,-0.019048,0.027894,0.000000,0.027894,1,9.449903e-01
4,0.013571,0.030357,0.013571,0.030357,0.043928,1,9.140737e-01
5,-0.025614,0.006611,0.000000,0.006611,0.006611,1,9.868223e-01
6,0.039184,-0.013902,0.039184,0.000000,0.039184,0,1.535423e-03
7,-0.010555,-0.015967,0.000000,0.000000,0.000000,0,0.000000e+00
8,0.019841,-0.016189,0.019841,0.000000,0.019841,0,3.936458e-04
9,-0.010731,-0.011023,0.000000,0.000000,0.000000,0,0.000000e+00


In [17]:
max_iterations = 1000
learning_rate = 0.01

loss_new = 0
counter = 0

for counter in range(max_iterations):
    grad_w1 = np.zeros(54)
    grad_w2 = np.zeros(54)
    grad_b1 = 0
    grad_b2 = 0
    
    for row in range(len(result)):
        error = result.loc[row, 'Preds'] - df_target.iloc[row]
        
        d_loss_d_pred = 2 * error
        
        # Gradient flows through the sum equally to both ReLU outputs
        d_pred_d_relu1 = 1
        d_pred_d_relu2 = 1
        
        d_relu1_d_linear1 = 1 if result.loc[row, 'Linear1'] > 0 else 0
        d_relu2_d_linear2 = 1 if result.loc[row, 'Linear2'] > 0 else 0
        
        grad_from_relu1 = d_loss_d_pred * d_pred_d_relu1 * d_relu1_d_linear1
        grad_w1 += grad_from_relu1 * df_inputs.iloc[row].values
        grad_b1 += grad_from_relu1
        
        grad_from_relu2 = d_loss_d_pred * d_pred_d_relu2 * d_relu2_d_linear2
        grad_w2 += grad_from_relu2 * df_inputs.iloc[row].values
        grad_b2 += grad_from_relu2
    
    grad_w1 /= len(result)
    grad_w2 /= len(result)
    grad_b1 /= len(result)
    grad_b2 /= len(result)
    
    values[0] -= learning_rate * grad_w1
    values[1] -= learning_rate * grad_w2
    biases[0] -= learning_rate * grad_b1
    biases[1] -= learning_rate * grad_b2
    
    result['Linear1'] = (df_inputs * values[0]).sum(axis=1) + biases[0]
    result['Linear2'] = (df_inputs * values[1]).sum(axis=1) + biases[1]
    result['ReLU1'] = result['Linear1'].apply(lambda x: max(0, x))
    result['ReLU2'] = result['Linear2'].apply(lambda x: max(0, x))
    result['Preds'] = result['ReLU1'] + result['ReLU2']
    result['Loss'] = (result['Preds'] - df_target)**2
    loss_new = result['Loss'].mean()
    
    if counter % 10 == 0:
        print(f'Iteration: {counter}, Loss: {loss_new:.6f}')
    
    if abs(loss_new - loss_last) < 1e-7:
        print(f'Converged at iteration {counter}')
        break
    
    loss_last = loss_new

print(f'Final loss: {loss_new}')

Iteration: 0, Loss: 0.513504
Iteration: 10, Loss: 0.187988
Iteration: 20, Loss: 0.130553
Iteration: 30, Loss: 0.096038
Iteration: 40, Loss: 0.073853
Iteration: 50, Loss: 0.058957
Iteration: 60, Loss: 0.048572
Iteration: 70, Loss: 0.041114
Iteration: 80, Loss: 0.035586
Iteration: 90, Loss: 0.031372
Iteration: 100, Loss: 0.028082
Iteration: 110, Loss: 0.025462
Iteration: 120, Loss: 0.023342
Iteration: 130, Loss: 0.021603
Iteration: 140, Loss: 0.020145
Iteration: 150, Loss: 0.018895
Iteration: 160, Loss: 0.017811
Iteration: 170, Loss: 0.016865
Iteration: 180, Loss: 0.016027
Iteration: 190, Loss: 0.015276
Iteration: 200, Loss: 0.014596
Iteration: 210, Loss: 0.013984
Iteration: 220, Loss: 0.013425
Iteration: 230, Loss: 0.012910
Iteration: 240, Loss: 0.012435
Iteration: 250, Loss: 0.011994
Iteration: 260, Loss: 0.011586
Iteration: 270, Loss: 0.011207
Iteration: 280, Loss: 0.010853
Iteration: 290, Loss: 0.010522
Iteration: 300, Loss: 0.010213
Iteration: 310, Loss: 0.009925
Iteration: 320, Los

In [18]:
result

,Linear1,Linear2,ReLU1,ReLU2,Preds,Actual,Loss
0,-0.046556,-0.040476,0.000000,0.000000,0.000000,0,0.000000e+00
1,0.532466,0.564996,0.532466,0.564996,1.097462,1,9.498791e-03
2,-0.021889,-0.036374,0.000000,0.000000,0.000000,0,0.000000e+00
3,0.424521,0.398746,0.424521,0.398746,0.823267,1,3.123439e-02
4,0.488165,0.503102,0.488165,0.503102,0.991267,1,7.626172e-05
5,0.459900,0.487352,0.459900,0.487352,0.947252,1,2.782400e-03
6,0.044101,0.000866,0.044101,0.000866,0.044967,0,2.022037e-03
7,0.017634,0.005470,0.017634,0.005470,0.023104,0,5.337793e-04
8,-0.020611,-0.057889,0.000000,0.000000,0.000000,0,0.000000e+00
9,-0.083192,-0.085453,0.000000,0.000000,0.000000,0,0.000000e+00


In [19]:
values

array([[-6.90430668e-02,  1.26427544e-01,  1.00680705e-01,
         9.39746908e-02,  7.74507614e-02, -1.20631141e-01,
         1.28592466e-05, -1.38138196e-01,  4.28751268e-02,
         4.75109193e-02,  4.86005848e-02,  3.47551021e-02,
         2.94381040e-02, -2.45507022e-01,  5.02267131e-03,
        -1.81024328e-01,  6.18735518e-03,  1.19909408e-02,
         3.24093373e-02,  4.19889175e-02,  5.40618796e-02,
         6.51178375e-02, -3.79776063e-02, -2.02655167e-02,
        -2.41401464e-03, -7.37503104e-03, -1.08991213e-02,
        -9.06110344e-03, -7.21347396e-03,  1.05704785e-02,
         6.74451846e-03,  2.11813359e-03,  2.66958525e-02,
         2.29136967e-02,  9.29243523e-04, -4.80893606e-03,
         6.69773934e-05,  4.00717499e-02, -3.34074005e-02,
        -4.41724395e-02,  5.58783964e-02,  4.76074512e-02,
        -6.99700445e-03, -3.00714600e-03,  7.44328200e-02,
        -1.54111313e-02,  1.33254904e-02,  1.97276833e-02,
        -9.48250834e-03, -1.35955815e-02,  4.50665442e-0